In [1]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from models import DQN, DuelingDQN
from replay_buffer import ReplayBuffer
from train import ConfigDQN, actualizar_modelo, entrenar_dqn
from evaluation import evaluar_modelo

SEMILLA = 42
N_ACCIONES = 6

np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Dispositivo seleccionado: {device}")
print(f"Número de acciones: {N_ACCIONES}")
print(f"Semilla: {SEMILLA}")

Dispositivo seleccionado: mps
Número de acciones: 6
Semilla: 42


In [2]:
config_extendido = ConfigDQN(
    nombre_experimento="v1_dqn_vanilla_extendido",
    total_pasos=1_000_000,  # el doble del original
)

resultado_extendido = entrenar_dqn(
    config=config_extendido,
    clase_modelo=DQN,
    device=device,
    usar_double_dqn=False,
    ruta_checkpoint_inicial="../models/v1_dqn_vanilla/checkpoint_final.pt",
)

A.L.E: Arcade Learning Environment (version 0.10.1+6a7e0ae)
[Powered by Stella]


Reanudando entrenamiento desde el paso 500,000 (checkpoint: ../models/v1_dqn_vanilla/checkpoint_final.pt)
Paso 501,000/1,000,000 | episodio=3 | epsilon=0.100 | loss=0.0055 | Q=2.291
Paso 502,000/1,000,000 | episodio=7 | epsilon=0.100 | loss=0.0033 | Q=2.323
Paso 503,000/1,000,000 | episodio=11 | epsilon=0.100 | loss=0.0039 | Q=2.286
Paso 504,000/1,000,000 | episodio=16 | epsilon=0.100 | loss=0.0042 | Q=2.119
Paso 505,000/1,000,000 | episodio=21 | epsilon=0.100 | loss=0.0198 | Q=2.503
Paso 506,000/1,000,000 | episodio=25 | epsilon=0.100 | loss=0.0039 | Q=2.113
Paso 507,000/1,000,000 | episodio=28 | epsilon=0.100 | loss=0.0071 | Q=1.946
Paso 508,000/1,000,000 | episodio=33 | epsilon=0.100 | loss=0.0024 | Q=2.111
Paso 509,000/1,000,000 | episodio=38 | epsilon=0.100 | loss=0.0168 | Q=2.253
Paso 510,000/1,000,000 | episodio=41 | epsilon=0.100 | loss=0.0034 | Q=2.130
Paso 511,000/1,000,000 | episodio=45 | epsilon=0.100 | loss=0.0086 | Q=2.135
Paso 512,000/1,000,000 | episodio=49 | epsilon=0.

In [3]:
ruta_mejor_extendido = "../models/v1_dqn_vanilla_extendido/mejor_modelo.pt"

checkpoint_extendido = torch.load(
    ruta_mejor_extendido,
    map_location=device,
    weights_only=True,
)

mejor_dqn_extendido = DQN(
    n_acciones=N_ACCIONES
).to(device)

mejor_dqn_extendido.load_state_dict(
    checkpoint_extendido["modelo_online_state_dict"]
)

mejor_dqn_extendido.eval()

print(
    f"Checkpoint cargado desde el paso: "
    f"{checkpoint_extendido['paso']:,}"
)

Checkpoint cargado desde el paso: 900,000


In [4]:
N_EPISODIOS_EVALUACION = 30
SEMILLA_BASE_EVALUACION = 42  

print(
    f"\nEvaluando DQN vanilla extendido durante "
    f"{N_EPISODIOS_EVALUACION} episodios...\n"
)

resultados_extendido, resumen_extendido = evaluar_modelo(
    modelo=mejor_dqn_extendido,
    config=config_extendido,
    device=device,
    n_episodios=N_EPISODIOS_EVALUACION,
    seed_base=SEMILLA_BASE_EVALUACION,
)

df_dqn_extendido = pd.DataFrame(resultados_extendido)
df_dqn_extendido["agente"] = "DQN vanilla extendido"

df_dqn_extendido = df_dqn_extendido[
    [
        "agente",
        "episodio",
        "seed",
        "recompensa_total",
        "pasos",
        "terminated",
        "truncated",
    ]
]

print("RESUMEN DE 30 EPISODIOS")

for metrica, valor in resumen_extendido.items():
    print(f"{metrica}: {valor:.2f}")

display(df_dqn_extendido.head(10))


Evaluando DQN vanilla extendido durante 30 episodios...

RESUMEN DE 30 EPISODIOS
promedio: 424.83
mediana: 425.00
desviacion: 105.63
minimo: 215.00
maximo: 715.00


,agente,episodio,seed,recompensa_total,pasos,terminated,truncated
0,DQN vanilla extendido,0,42,515.0,975,True,False
1,DQN vanilla extendido,1,43,440.0,590,True,False
2,DQN vanilla extendido,2,44,470.0,594,True,False
3,DQN vanilla extendido,3,45,270.0,514,True,False
4,DQN vanilla extendido,4,46,520.0,853,True,False
5,DQN vanilla extendido,5,47,385.0,603,True,False
6,DQN vanilla extendido,6,48,350.0,543,True,False
7,DQN vanilla extendido,7,49,415.0,691,True,False
8,DQN vanilla extendido,8,50,275.0,439,True,False
9,DQN vanilla extendido,9,51,425.0,624,True,False


## Guardar y comparar con iteraciones anteriores

In [6]:
from pathlib import Path 

ruta_resultados_extendido = Path(
    "../logs/evaluacion_dqn_vanilla_extendido.csv"
)

df_dqn_extendido.to_csv(
    ruta_resultados_extendido,
    index=False,
)

df_baseline = pd.read_csv("../logs/baseline_episodios.csv")
df_dqn_vanilla = pd.read_csv("../logs/evaluacion_dqn_vanilla.csv")
df_double_dqn = pd.read_csv("../logs/evaluacion_double_dqn.csv")
df_dueling_dqn = pd.read_csv("../logs/evaluacion_dueling_double_dqn.csv")

columnas_comparacion = [
    "agente",
    "episodio",
    "seed",
    "recompensa_total",
    "pasos",
    "terminated",
    "truncated",
]

df_comparacion_modelos = pd.concat(
    [
        df_baseline[columnas_comparacion],
        df_dqn_vanilla[columnas_comparacion],
        df_double_dqn[columnas_comparacion],
        df_dueling_dqn[columnas_comparacion],
        df_dqn_extendido[columnas_comparacion],
    ],
    ignore_index=True,
)

resumen_comparacion_modelos = (
    df_comparacion_modelos
    .groupby("agente")
    .agg(
        episodios=("recompensa_total", "count"),
        promedio=("recompensa_total", "mean"),
        mediana=("recompensa_total", "median"),
        desviacion=("recompensa_total", "std"),
        minimo=("recompensa_total", "min"),
        maximo=("recompensa_total", "max"),
        pasos_promedio=("pasos", "mean"),
    )
    .round(2)
    .reset_index()
)

display(resumen_comparacion_modelos)

#comparación pareada: el extendido contra cada una de las tres iteraciones anteriores
comparacion_rl = (
    pd.concat(
        [
            df_dqn_vanilla,
            df_double_dqn,
            df_dueling_dqn,
            df_dqn_extendido,
        ],
        ignore_index=True,
    )
    .pivot(
        index="seed",
        columns="agente",
        values="recompensa_total",
    )
    .reset_index()
)

pares_a_comparar = [
    ("DQN vanilla", "DQN vanilla extendido"),
    ("Double DQN", "DQN vanilla extendido"),
    ("Dueling Double DQN", "DQN vanilla extendido"),
]

for agente_a, agente_b in pares_a_comparar:
    diferencia = comparacion_rl[agente_b] - comparacion_rl[agente_a]

    print(f"\n{agente_a}  vs.  {agente_b}")
    print(f"Victorias {agente_a}: {(diferencia < 0).sum()}")
    print(f"Victorias {agente_b}: {(diferencia > 0).sum()}")
    print(f"Empates: {(diferencia == 0).sum()}")

print(f"\nResultados guardados en: {ruta_resultados_extendido}")

,agente,episodios,promedio,mediana,desviacion,minimo,maximo,pasos_promedio
0,Aleatorio,30,165.67,137.5,99.26,35.0,485.0,524.03
1,DQN vanilla,30,406.67,377.5,139.11,215.0,670.0,705.80
2,DQN vanilla extendido,30,424.83,425.0,107.43,215.0,715.0,673.00
3,Double DQN,30,283.33,262.5,154.85,105.0,925.0,614.73
4,Dueling Double DQN,30,263.33,215.0,112.84,155.0,580.0,597.50
5,Regla simple,30,211.50,190.0,95.49,55.0,360.0,569.33



DQN vanilla  vs.  DQN vanilla extendido
Victorias DQN vanilla: 12
Victorias DQN vanilla extendido: 18
Empates: 0

Double DQN  vs.  DQN vanilla extendido
Victorias Double DQN: 6
Victorias DQN vanilla extendido: 24
Empates: 0

Dueling Double DQN  vs.  DQN vanilla extendido
Victorias Dueling Double DQN: 2
Victorias DQN vanilla extendido: 27
Empates: 1

Resultados guardados en: ../logs/evaluacion_dqn_vanilla_extendido.csv
